# Spliv: check your split before training
Run every cell in order. This notebook loads scikit-learn's bundled breast cancer dataset, demonstrates a contaminated split, checks a proper split, and then fits a preprocessing + classification pipeline.

This is a software example, not a clinical model or a medical recommendation. A clean Spliv report only means the requested checks found no issues; it does not certify an evaluation protocol.

**Where it fits:** load data → split → **audit** → fit preprocessing on training data → fit model → evaluate once on the holdout.

Spliv makes no network calls to audit data. In Colab, data you upload is processed in Google's runtime. Keep sensitive data in an environment you are authorized to use.


In [ ]:
%pip install -q "spliv[notebook]==0.2.0"

In [ ]:
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from spliv import audit, AuditFailed

df = load_breast_cancer(as_frame=True).frame
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["target"])
print(f"Training rows: {len(train)} | Test rows: {len(test)}")


## 1. See a real failure
An accidental concatenation puts a training observation into the test set. Spliv should catch that even though the DataFrame indexes were reset. It reports row positions, never raw feature values.


In [ ]:
contaminated_test = pd.concat([test, train.iloc[[0]]], ignore_index=True)
bad_report = audit(train, contaminated_test, target="target")
display(pd.DataFrame(bad_report.summary()))
try:
    bad_report.raise_for_findings()
except AuditFailed:
    print("Training blocked: review the split before fitting.")
else:
    raise AssertionError("The deliberately contaminated split was not blocked")


## 2. Audit the original split and view the report
Pass your DataFrames directly. `target` excludes the label from feature matching and enables the exact target-copy check. The display is an isolated, scrollable report inside the notebook.


In [ ]:
report = audit(train, test, target="target")
display(report)
report.raise_for_findings()  # raises AuditFailed before model.fit if the gate fails


## 3. Train only after the gate passes
Spliv does not inspect preprocessing code. The pipeline below fits scaling on training data only. Use cross-validation within the training set for model selection; avoid repeatedly tuning against this holdout.


In [ ]:
X_train, y_train = train.drop(columns="target"), train["target"]
X_test, y_test = test.drop(columns="target"), test["target"]
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42))
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(f"Example holdout accuracy: {accuracy_score(y_test, predictions):.3f}")


## 4. Save a reviewable artifact
The HTML is standalone. The JSON contains counts, labels, and record positions, without raw cell values. Review filenames and column labels before sharing.


In [ ]:
import json
from pathlib import Path
Path("split-report.html").write_text(report.to_html(), encoding="utf-8")
Path("split-report.json").write_text(json.dumps(report.to_dict(), indent=2), encoding="utf-8")
print("Saved split-report.html and split-report.json")


## Use your own data
Upload a CSV using Colab's Files sidebar. Replace the dataset-loading cell with:

```python
df = pd.read_csv("your-data.csv")
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["your_label"])
report = audit(train, test, target="your_label", ignore=("record_id",))
display(report)
report.raise_for_findings()
```

Use stratification for classification labels, not continuous regression targets. Random row splits suit only independent observations. For repeated customers/patients use an entity split and `group="customer_id"`; for forecasting use chronological splits and `time="observed_at"`. See [practical recipes](https://github.com/zorox06/holdout-audit/blob/main/docs/workflows.md).

DataFrame indexes are ignored. Convert an index you need checked into a named column first. Record numbers are **1-based positions**, so inspect `train.iloc[record_number - 1]`, not `.loc`. Drop target, ID, entity, and time metadata from the model features as appropriate; Spliv's exclusions do not change your training data.

DataFrames use the same exact-string contract as `to_csv(index=False)`: missing cells become empty strings, `1` and `1.0` differ, and no type normalization occurs. Match intended dtypes across splits before auditing. Flat unique string column names and scalar cells are required.

To download the HTML from Colab, run this in a new cell:

```python
from google.colab import files
files.download("split-report.html")
```

Do not delete flagged rows automatically or silence a gate solely to obtain a clean score. Decide whether each finding conflicts with your evaluation goal. `report.raise_for_findings("error")` permits complete warning-level findings; incomplete requested checks still block training.
